# Check evidence packets

Change `packet_id`, then restart the kernel and run all cells.
Run the setup cell first; later cells use its variables.
Nothing here changes files or calls a model.

Sources are extracted from PDFs in `private/pdfs/` into `.txt` and `.json` files in `private/sources/`. The JSON records the PDF filename, hashes, extraction details, and page spans. Apple uses `SRC_01`; Microsoft uses `SRC_02`.

In [26]:
import json
from pathlib import Path
from decimal import Decimal
from pprint import pprint

# Find the repo even when Jupyter starts in src/ or a notebook folder.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/benchmark/private/packets").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from inside the agent_test repository.")
folder = repo / "data/benchmark"
packet_id = "EP_0010"  # EP_0001 = Apple, EP_0002 = Microsoft

packet = json.loads((folder / f"private/packets/{packet_id}.json").read_text(encoding="utf-8"))
masked_packet = json.loads((folder / f"public/evidence_packets/{packet_id}.json").read_text(encoding="utf-8"))
source_id = packet["source_id"]
source = (folder / f"private/sources/{source_id}.txt").read_text(encoding="utf-8")
source_info = json.loads((folder / f"private/sources/{source_id}.json").read_text(encoding="utf-8"))

print("Company:", packet["company"]["name"])
print("Original source:", source_info.get("input_file") or source_info.get("source_url"))
print("Saved text:", folder / f"private/sources/{source_id}.txt")

Company: UnitedHealth Group Incorporated
Original source: private/pdfs/unh-20251231.pdf
Saved text: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/private/sources/SRC_10.txt


## The evidence packet

`private` has the original names and source references. `public` has the masked version the agents will see.

In [27]:
pprint(packet, sort_dicts=False)

{'schema_version': '0.1',
 'packet_id': 'EP_0010',
 'company_alias': 'COMPANY_10',
 'company': {'name': 'UnitedHealth Group Incorporated', 'ticker': 'UNH'},
 'fiscal_year': 2025,
 'filing': {'form': '10-K',
            'accession_number': '0000731766-26-000062',
            'filing_date': '2026-03-02',
            'source_url': 'https://www.sec.gov/Archives/edgar/data/731766/000073176626000062/unh-20251231.htm'},
 'source_id': 'SRC_10',
 'evidence': [{'evidence_id': 'EP_0010_E01',
               'section': 'Item 7',
               'text': 'RESULTS SUMMARY\n'
                       'The following table summarizes our consolidated '
                       'results of operations and other financial '
                       'information:\n'
                       '(in millions, except percentages and per share data)\n'
                       'For the Years Ended December 31, Change\n'
                       '2025 2024 2023 2025 vs. 2024\n'
                       'Revenues:\n'
             

## Source text vs stored evidence

`source_span` is `[start, end]` in the saved source text. The end character is excluded.
Each comparison should print `Equal: True`. Use the printed locator to check the original filing too.

In [28]:
for evidence in packet["evidence"]:
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]

    print("\n---", evidence["evidence_id"], "---")
    print("Location:", evidence["locator"])
    print("Source span:", [start, end])
    print("\nFROM SOURCE:\n", source_excerpt)
    print("\nIN PACKET:\n", evidence["text"])
    print("\nEqual:", source_excerpt == evidence["text"])


--- EP_0010_E01 ---
Location: PDF pages [30]; Consolidated results table including fiscal years, units and ratio definitions; printed p. 28
Source span: [148745, 153008]

FROM SOURCE:
 RESULTS SUMMARY
The following table summarizes our consolidated results of operations and other financial information:
(in millions, except percentages and per share data)
For the Years Ended December 31, Change
2025 2024 2023 2025 vs. 2024
Revenues:
Premiums   ..................................................................................... $ 352,229 $ 308,810 $ 290,827 $ 43,419  14% 
Products    .......................................................................................  53,380  50,226  42,583  3,154  6 
Services    .......................................................................................  38,038  36,040  34,123  1,998  6 
Investment and other income  ......................................................  3,920  5,202  4,089  (1,282)  (25) 
Total revenues    ............

## Source numbers vs stored numbers

Find each recorded source number in the original excerpt and compare it with the stored value.
Also check the printed context: a matching number alone does not prove it belongs to the right row, year, or unit.

In [29]:
for fact in packet["facts"]:
    evidence = next(e for e in packet["evidence"] if e["evidence_id"] == fact["evidence_id"])
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]
    position = source_excerpt.find(fact["source_value"])
    source_number = source_excerpt[position:position + len(fact["source_value"])] if position >= 0 else None

    print("\n---", fact["fact_id"], fact["metric"], "---")
    print("Evidence:", fact["evidence_id"], "|", fact["locator"])
    print("Period:", fact["period"], "| Unit:", fact["unit"])
    if position >= 0:
        print("Source context:\n", source_excerpt[max(0, position - 100):position + len(fact["source_value"]) + 100])
    print("Source number:", source_number)
    print("Packet number:", fact["value"])
    equal = source_number is not None and Decimal(source_number.replace(",", "")) == Decimal(fact["value"])
    print("Equal:", equal)


--- EP_0010_F01 revenue ---
Evidence: EP_0010_E01 | Total revenues; FY2025 column
Period: {'label': 'FY2025', 'start': '2025-01-01', 'end': '2025-12-31'} | Unit: USD_million
Source context:
 revenues    ......................................................................................  447,567  400,278  371,622  47,289  12 
Operating costs:
Medical costs    ..................................
Source number: 447,567
Packet number: 447567
Equal: True

--- EP_0010_F02 revenue ---
Evidence: EP_0010_E01 | Total revenues; FY2024 column
Period: {'label': 'FY2024', 'start': '2024-01-01', 'end': '2024-12-31'} | Unit: USD_million
Source context:
    ......................................................................................  447,567  400,278  371,622  47,289  12 
Operating costs:
Medical costs    ...........................................
Source number: 400,278
Packet number: 400278
Equal: True

--- EP_0010_F03 operating_income ---
Evidence: EP_0010_E01 | Earnings from operations

## Original vs masked

Read each pair. Names and brands should change; the numbers and meaning should stay the same.

In [30]:
for evidence in packet["evidence"]:
    masked = next(e for e in masked_packet["evidence"] if e["evidence_id"] == evidence["evidence_id"])
    print("\n---", evidence["evidence_id"], "---")
    print("ORIGINAL:\n", evidence["text"])
    print("\nMASKED:\n", masked["text"])

print("\nStored numbers unchanged by masking:")
for fact in packet["facts"]:
    masked_fact = next(f for f in masked_packet["facts"] if f["fact_id"] == fact["fact_id"])
    print(fact["fact_id"], fact["value"], "->", masked_fact["value"], "Equal:", fact["value"] == masked_fact["value"])


--- EP_0010_E01 ---
ORIGINAL:
 RESULTS SUMMARY
The following table summarizes our consolidated results of operations and other financial information:
(in millions, except percentages and per share data)
For the Years Ended December 31, Change
2025 2024 2023 2025 vs. 2024
Revenues:
Premiums   ..................................................................................... $ 352,229 $ 308,810 $ 290,827 $ 43,419  14% 
Products    .......................................................................................  53,380  50,226  42,583  3,154  6 
Services    .......................................................................................  38,038  36,040  34,123  1,998  6 
Investment and other income  ......................................................  3,920  5,202  4,089  (1,282)  (25) 
Total revenues    ......................................................................................  447,567  400,278  371,622  47,289  12 
Operating costs:
Medical costs    .....

## Notes

Keep any problems found here in the matching report review notebook,
for example `private/reviews/EP_0001_review.ipynb`.
That one notebook holds the evidence checks, report feedback, edits and final output.